# LLM2Seq Training on Kaggle / Colab

Notebook này được thiết kế để chạy trực tiếp trên **Kaggle** (với GPU P100/T4x2/L4) hoặc **Google Colab** (T4/V100/A100).

## Hướng dẫn:
1. Do LLM2Seq có kiến trúc gồm rất nhiều files (`src/models/`, `src/training/`...), nên bạn không thể copy paste vào 1 cell như code cũ.
2. **Cách 1 (Kaggle)**: Nén toàn bộ project trên máy của bạn thành `llm2seq.zip`, tạo thành 1 Kaggle Dataset rồi Add Data vào notebook. Sau đó copy đè sang `/kaggle/working/`.
3. **Cách 2 (Colab)**: Upload file `llm2seq.zip` trực tiếp vào thanh bên trái của Colab rồi giải nén, hoặc kết nối Google Drive, hoặc Clone từ Github của bạn.

Ở notebook này, chúng ta giả định bạn đã upload `llm2seq.zip` lên môi trường chạy.

### 1. Chuẩn bị Môi trường & Code

In [ ]:
# Setup Environment (Tương thích P100/T4)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!pip uninstall -y -q torch torchvision torchaudio
!pip install -q --index-url https://download.pytorch.org/whl/cu118 torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1
!pip install -q transformers==4.46.3 datasets evaluate accelerate peft rouge-score bert-score sacrebleu pyyaml


In [ ]:
# GIẢI NÉN CODE (Nếu bạn upload llm2seq.zip)
# !unzip -q llm2seq.zip -d .

# Nếu bạn clone từ github thì:
# !git clone <đường dẫn github của bạn>
# %cd <tên repo của bạn>

import sys
if "." not in sys.path:
    sys.path.insert(0, ".")


### 2. Tiền xử lý Dữ liệu
Chạy script có sẵn để tải dataset từ HuggingFace và convert sang định dạng `JSONL` mà LLM2Seq yêu cầu.

In [ ]:
!python -m llm2seq.src.data.preprocess \
    --dataset nam194/vietnews \
    --output_dir llm2seq/data/processed \
    --source_field article \
    --target_field abstract \
    --max_train 4000 \
    --max_eval 500

### 3. Huấn luyện (Training)
LLM2Seq được thiết kế hoàn toàn qua file cấu hình `YAML`. Bạn có thể chỉnh sửa `batch_size`, `lr`, `max_steps` ở các file `configs/*.yaml` rồi chạy các cell dưới đây tương ứng.

In [ ]:
# Tùy chọn 1: Train Baseline (Chỉ encoder + decoder thông thường, không KD, không MTP)
!python -m llm2seq.src.training.trainer --config llm2seq/configs/baseline.yaml

In [ ]:
# Tùy chọn 2: Train KD (Knowledge Distillation từ teacher)
# Lưu ý: Bạn cần có dữ liệu dạng teacher_logits hoặc teacher_targets
# !python -m llm2seq.src.training.trainer --config llm2seq/configs/kd_only.yaml

In [ ]:
# Tùy chọn 3: Train MTP (Multi-Token Prediction, tăng tốc inference)
# !python -m llm2seq.src.training.trainer --config llm2seq/configs/mtp_only.yaml

In [ ]:
# Tùy chọn 4: Train Full (KD + MTP)
# !python -m llm2seq.src.training.trainer --config llm2seq/configs/kd_mtp_full.yaml

### 4. Đánh giá & Suy luận (Inference)
Sau khi train xong, model sẽ được lưu tại thư mục `runs/`. Hãy thử load model lên và sinh thử văn bản.

In [ ]:
import torch
import yaml
import os
from transformers import AutoTokenizer
from llm2seq.src.models.llm2seq_model import LLM2Seq, LLM2SeqConfig
from llm2seq.src.inference.generate import autoregressive_generate
from llm2seq.src.inference.generate_mtp import mtp_generate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load lại cấu hình đã dùng để train
with open('llm2seq/configs/baseline.yaml', 'r') as f:
    cfg = LLM2SeqConfig(yaml.safe_load(f))

tokenizer = AutoTokenizer.from_pretrained(cfg.encoder_name, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Khởi tạo model
model = LLM2Seq(cfg, vocab_size=len(tokenizer))

# Load Checkpoint (Ví dụ model lưu trong thư mục runs/llm2seq_baseline)
ckpt_path = 'runs/llm2seq_baseline/best.pt'
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu')['model_state_dict'])
    print("✅ Đã load checkpoint thành công.")
else:
    print("⚠️ Không tìm thấy checkpoint. Model đang sử dụng random weights để test!")

model.to(device)
model.eval()

text = "Gia đình phát hiện căn hộ bị bẻ khóa, nhiều đồ đạc có giá trị đã không cánh mà bay. Vụ việc nhanh chóng được báo cho cảnh sát địa phương."
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    out_ids = autoregressive_generate(
        model=model,
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=30,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
    )

print("\n--- TEXT GỐC ---")
print(text)
print("\n--- TÓM TẮT ---")
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))